# Lab 4\. Devices: Checking, Placing, and Moving Tensors

## Estimated Time

25–35 minutes.

## How to Perform the Exercise

You will work entirely in one notebook, running cells from top to bottom. Create a new notebook in Google Colab (**File > New notebook**) or a new `.ipynb` in local Jupyter, and add each code block below as its own cell as you reach it. Read the explanation before running each cell, run it, and compare what you see against the Output shown here.

The rock-paper-scissors images come from the LFD2000 GitHub repository, which you clone in Step 2\. That repository holds the dataset under `LFD2000/datasets/RPS`, organized as one folder per class (rock, paper, and scissors). You do not modify any files in the dataset; you only read the images and turn them into tensors.

Commands in this lab use the notebook form, where a line beginning with `!` runs a shell command from inside a notebook cell (for example, `!pip install ...`). If you run these in a local terminal instead of a notebook cell, drop the leading `!`.

**Note:** If you are on Google Colab and want to work against a real GPU, enable it now, before you clone the dataset. Switching the runtime later restarts it and clears anything you have downloaded. To enable it, open **Runtime > Change runtime type**, choose a GPU accelerator, and save. Leaving it on CPU is completely fine; the lab is written to work either way.

\[SCREENSHOT\] of the Colab runtime-type dialog here if screenshots are used in this course.\]

---

## Environment Preparation (Setup)

The two steps in this section only stand up the environment: they install the libraries and download the dataset. They are setup, not the device skills the lab teaches, so keep them lean and move on to Step 3 once they succeed.

### Step 1\. Install PyTorch and TorchVision, and Import

You need PyTorch for tensors and device operations, and TorchVision for loading images as tensors. Install both, then import the pieces the lab uses. Installing in the first cell makes the notebook self-contained, so anyone who opens it later gets the right libraries.

In [ ]:
!pip install torch torchvision

Output

```
Successfully installed torch-2.12.1 torchvision-0.27.0
```

If the libraries are already present (common on Colab), pip reports that the requirement is already satisfied instead, which is equally fine.

Next, import the modules you will use throughout the lab. `torch` is the core library; `v2` is TorchVision's current transforms module; and `ImageFolder` loads images arranged one folder per class.

In [ ]:
import torch
from torchvision.transforms import v2
from torchvision.datasets import ImageFolder

print("PyTorch version:", torch.__version__)

Output

```
PyTorch version: 2.12.1
```

Confirm the version line reports a 2.12 release. The device behavior in this lab is the same across the 2.12 line.

### Step 2\. Get the Rock-Paper-Scissors Dataset

Download the dataset by cloning the LFD2000 repository. Cloning creates a directory named `LFD2000` in your working folder, with the images under `LFD2000/datasets/RPS`. You need the images on disk before you can load them as tensors in Step 4\.

In [ ]:
!git clone https://github.com/lftraining/LFD2000.git

Output

```
Cloning into 'LFD2000'...
remote: Enumerating objects: ...
Receiving objects: 100% ...
Resolving deltas: 100% ...
```

To confirm the dataset is where you expect, list the class folders under the dataset root. You should see one folder per class.

In [ ]:
import os
print(os.listdir("LFD2000/datasets/RPS"))

Output

```
['rock', 'paper', 'scissors']
```

Seeing the three class folders confirms the dataset root path is correct, which is the path you hand to `ImageFolder` in Step 4\.

---

## Working with Devices

The remaining steps are the core of the lab. You will inspect your environment, load real data as tensors, move tensors between devices, break the device rule on purpose, fix it, and finish with the reusable device-agnostic block.

### Step 3\. Check Which Devices Are Available

Before you place a tensor on a GPU, you have to know whether one exists. PyTorch gives you a direct check for each accelerator: `torch.cuda.is_available()` for an NVIDIA (CUDA) GPU, and `torch.backends.mps.is_available()` for Apple Silicon (MPS). The CPU is always present, so there is no check for it.

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())

Output (on a CPU-only environment, such as a default Colab runtime)

```
CUDA available: False
MPS available: False
```

Read your own result rather than the sample. On a machine with an NVIDIA GPU, or a Colab GPU runtime, the CUDA line reads `True`. On an Apple Silicon Mac, the MPS line reads `True`. Whatever you see here tells you which accelerators the rest of the lab can use on your machine.

### Step 4\. Load the Rock-Paper-Scissors Images as Tensors

Bring in real data so the device work applies to the course problem rather than to throwaway numbers. You will rebuild the same image-to-tensor pipeline from Chapter 3 and load the images with `ImageFolder`. This is the same pipeline you saw before, recreated here because this lab does not carry anything over from the Chapter 3 lab.

The pipeline chains three transforms with `v2.Compose`: `v2.ToImage()` turns a loaded image into a tensor, `v2.Resize((128, 128))` forces every image to the same height and width, and `v2.ToDtype(torch.float32, scale=True)` converts to PyTorch's default decimal type and rescales pixel values from the 0–255 range to 0.0–1.0.

In [ ]:
to_tensor = v2.Compose([
    v2.ToImage(),
    v2.Resize((128, 128)),
    v2.ToDtype(torch.float32, scale=True),
])

dataset = ImageFolder(root="LFD2000/datasets/RPS", transform=to_tensor)
print("Number of images:", len(dataset))
print("Classes:", dataset.classes)

Output

```
Number of images: 2188
Classes: ['paper', 'rock', 'scissors']
```

The image count will match whatever the dataset contains, and `ImageFolder` sorts the class names alphabetically, so do not worry if the ordering differs from how you listed the folders earlier. Now take a single image tensor out of the dataset and inspect it. Each item is a pair: the image tensor and its integer class label.

In [ ]:
image, label = dataset[0]
print("Shape:", image.shape)
print("Dtype:", image.dtype)
print("Device:", image.device)

Output

```
Shape: torch.Size([3, 128, 128])
Dtype: torch.float32
Device: cpu
```

The shape `(3, 128, 128)` is the image tensor layout from Chapter 3: three color channels, each 128 pixels high and 128 wide. The important line for this lab is the last one: the tensor reports `device: cpu`. Freshly loaded data always starts on the CPU, which is exactly why you will move it before running work on an accelerator.

### Step 5\. Move a Tensor to the Best Available Device

Choose a device to work on, then move the image tensor onto it. For this step, pick CUDA if it is available and fall back to the CPU otherwise, so the code runs anywhere. You move a tensor with `.to()`, which returns a **copy** on the target device.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

image_on_device = image.to(device)
print("Original image device:", image.device)
print("Moved copy device:", image_on_device.device)

Output (on a CPU-only environment)

```
Using device: cpu
Original image device: cpu
Moved copy device: cpu
```

On a GPU machine, `Using device` reads `cuda`, the original still reads `cpu`, and the moved copy reads `cuda:0`. That contrast is the key point: `.to()` does not change the original tensor in place. It hands back a new tensor on the target device, so you must assign the result to a variable to keep it. Writing `image.to(device)` on its own and then continuing to use `image` would leave you on the old device with nothing moved.

PyTorch also provides two shortcuts: `.cuda()` moves a tensor to a CUDA GPU and `.cpu()` moves it back to the CPU. They behave like `.to("cuda")` and `.to("cpu")`. There is no `.mps()` shortcut, so `.to()` is the one method that works for every device, which is why this lab uses it. \[Verified from docs: PyTorch 2.12 CUDA semantics — `to()` returns a copy on the target device; `.cuda()` / `.cpu()` shortcuts.\]

### Step 6\. Trigger the Device Mismatch Error on Purpose

The most common device error appears when a single operation combines tensors that live on different devices. Causing it deliberately, and reading the message, makes it easy to recognize later. This only happens when you actually have a second device to place a tensor on, so the code below runs the experiment when an accelerator is available and explains the situation when it is not. The `try`/`except` wrapper catches the error so your notebook keeps running.

In [ ]:
if device != "cpu":
    a = torch.tensor([1.0, 2.0, 3.0])                 # stays on the CPU
    b = torch.tensor([4.0, 5.0, 6.0], device=device)  # on the accelerator
    try:
        result = a + b
    except RuntimeError as e:
        print("RuntimeError:", e)
else:
    print("No accelerator available, so this mismatch cannot be triggered on CPU alone.")
    print("On a GPU or MPS machine you would see a RuntimeError like the one below:")
    print("Expected all tensors to be on the same device, "
          "but found at least two devices, cuda:0 and cpu!")

Output (on a machine with an accelerator)

```
RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!
```

Read what the message says. It names the exact problem, tensors on more than one device, and it names the two devices involved. Here that is `cuda:0` (where `b` lives) and `cpu` (where `a` lives). The two tensors met in a single addition without sharing a device, so PyTorch stopped instead of guessing which device you meant. If you are on a CPU-only environment, you cannot place `a` and `b` on different devices, so the code prints the message you would see on an accelerator instead.

### Step 7\. Fix the Mismatch by Aligning Devices

The fix is to put both tensors on the same device before combining them. Move the CPU tensor onto the accelerator so it matches the other one, then repeat the operation. On a CPU-only environment there is nothing to fix, because both tensors are already on the CPU, so the code guards for that case.

In [ ]:
if device != "cpu":
    a = a.to(device)          # move a onto the same device as b
    result = a + b            # both on the accelerator now
    print("Result:", result)
    print("Result device:", result.device)
else:
    a = torch.tensor([1.0, 2.0, 3.0])
    b = torch.tensor([4.0, 5.0, 6.0])
    result = a + b            # both already on the CPU
    print("Result:", result)
    print("Result device:", result.device)

Output (on a CPU-only environment)

```
Result: tensor([5., 7., 9.])
Result device: cpu
```

On an accelerator, `Result device` reads `cuda:0`, because the result of an operation lands on the shared device of its inputs. You could have fixed the mismatch the other way, by moving `b` down to the CPU with `b = b.cpu()`; what matters is that the two tensors agree on a device before they meet. There is no single correct direction; you move whichever tensor needs to join the other.

### Step 8\. Write the Device-Agnostic Selection Block

You now have every piece needed for device-agnostic code: code that runs on whatever hardware is present without editing. The block below checks for CUDA first, then MPS, then falls back to the CPU, and stores the result in one `device` variable. Routing every tensor through that variable keeps them together, so the mismatch from Step 6 cannot happen.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

# Route the rock-paper-scissors image tensor through the chosen device
image, label = dataset[0]
image = image.to(device)
print("Image device:", image.device)

Output (on a CPU-only environment)

```
Using device: cpu
Image device: cpu
```

On a GPU machine both lines report `cuda`, on an Apple Silicon Mac both report `mps`, and the same cell runs correctly in every case. `torch.device("cuda")` builds a device object that behaves like the string `"cuda"` when passed to `.to()`. This block is the reusable pattern to keep. You will use exactly this selection in the next chapter to place both a model and its training data on one device before training.

---

## Cleanup

This lab is not cumulative, so you can return your environment to its pre-lab state. Nothing you created here is needed by a later chapter; Chapter 5 acquires the dataset again in its own notebook.

The only on-disk artifact is the cloned dataset directory. Remove it when you are done.

**Warning:** This permanently deletes the `LFD2000` directory and everything in it. Run it only when you have finished the lab.

In [ ]:
!rm -rf LFD2000

The in-notebook Python objects (the `to_tensor` pipeline, the `dataset`, and the tensors you created) exist only in the notebook's memory. On Google Colab they clear automatically when the runtime disconnects. On a local machine, closing the notebook and shutting down its kernel releases them; the installed `torch` and `torchvision` packages remain until you remove them, which you do not need to do for this course. Cleanup here is complete: it restores the environment to its pre-lab state, and no state carries forward.